# Component 1 — Acoustic Robustness & Quality Gate

Component 1 is the **first stage** of the PD speech-screening pipeline. It is a
**quality gate, not a PD classifier**. For every recording it:

1. checks the recording is usable (length, clipping, voice present, noise level),
2. tries to clean the noise,
3. checks the cleaning did **not** damage the voice biomarkers Component 2 relies on
   (jitter, shimmer, HNR, pitch),
4. decides **PASS** (forward the audio to Component 2 / 3) or **FAIL** (ask the user to re-record).

This notebook walks through that process step by step using the component's own code
(`component1_robustness/*.py`). It does not re-implement anything, so what you see here
is exactly what the CLI, the API and the screening pipeline do.

| Section | What it shows |
|---|---|
| 1 | Setup and configuration |
| 2 | Input recording |
| 3 | Duration check |
| 4 | Quality measurements before cleaning |
| 5 | Enhancement (noise cleaning) |
| 6 | Biomarker preservation check |
| 7 | Gate decision |
| 8 | Full pipeline in one call + saved outputs |
| 9 | Hand-off to Component 2 (sample-rate check) |
| 10 | Behaviour under different recording conditions |
| 11 | Evidence for the 10 dB threshold |
| 12 | Personal recordings (inference only) |

## 1. Setup and configuration

Run this notebook with the project's `.venv` as the kernel. It needs Jupyter
(`pip install notebook ipykernel`) in addition to `requirements.txt`.

The cell below finds the project root (`parkinson-speech-ai/`) wherever the notebook
is opened from, so the project's packages can be imported.

In [ ]:
from pathlib import Path
import os
import sys
import warnings

def find_project_root(start=None):
    """Walk up from the notebook folder until configs/component1.yaml is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "configs" / "component1.yaml").is_file():
            return folder
    raise FileNotFoundError("Could not find parkinson-speech-ai/ (configs/component1.yaml). "
                            "Open this notebook from inside the project.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, Image, display

from component1_robustness.config import load_config, resolve_path
from component1_robustness.pipeline import TASK_FEATURES, TASKS, load_standardised, run_gate
from component1_robustness.quality_metrics import check_duration
from component1_robustness.gate_models import gate_inputs, load_gate_model
from component1_robustness.enhancement import enhance
from component1_robustness.features_check import compare_features
from component1_robustness.visualise import plot_spectrograms, plot_waveforms
from component1_robustness.degradation import add_white_noise, apply_gain

print("Project root:", PROJECT_ROOT)

In [ ]:
CONFIG = load_config()   # configs/component1.yaml — fixed by the developer, never by the end user

settings = {
    "config_version": CONFIG["config_version"],
    "seed": CONFIG["seed"],
    "valid duration (s)": f'{CONFIG["min_duration_s"]} - {CONFIG["max_duration_s"]}',
    "SNR/HNR pass threshold (dB)": CONFIG["reliable_db"],
    "max clipping (%)": CONFIG["max_clipping_pct"],
    "min voice activity (%)": CONFIG["min_voice_activity_pct"],
    "enhancement method": CONFIG["enhancement_method"],
    "gate model": CONFIG["gate_model"],
    "feature tolerances (vowel)": CONFIG["feature_tolerances"],
    "feature tolerances (ddk)": CONFIG["ddk_feature_tolerances"],
}
display(pd.Series(settings, name="value").to_frame())

GATE = load_gate_model(CONFIG["gate_model"], CONFIG)
print("Gate model loaded:", GATE.name)

## 2. Input recording

The input is **one recording** (`.wav`, `.m4a` or `.mp3`) and its **task**:

- `vowel` — a sustained "aaaa", forwarded to **Component 2** (phonation)
- `ddk` — "pa-ta-ka", forwarded to **Component 3** (speech-motor)

Change `INPUT_PATH` / `TASK` to try another recording.

In [ ]:
INPUT_PATH = PROJECT_ROOT / "data" / "test_audio" / "P002_A_01.m4a"
TASK = "vowel"

assert INPUT_PATH.is_file(), f"Recording not found: {INPUT_PATH}"
assert TASK in TASKS, f"TASK must be one of {list(TASKS)}"

# Same loader the pipeline uses: any format -> mono, at the recording's own sample rate.
y_raw, sr = load_standardised(INPUT_PATH)

print(f"File        : {INPUT_PATH.name}")
print(f"Task        : {TASK}  ->  {TASKS[TASK]}")
print(f"Sample rate : {sr} Hz")
print(f"Duration    : {len(y_raw) / sr:.2f} s")
display(Audio(y_raw, rate=sr))

t = np.arange(len(y_raw)) / sr
plt.figure(figsize=(11, 3))
plt.plot(t, y_raw, linewidth=0.5)
plt.xlabel("Time (s)"); plt.ylabel("Amplitude"); plt.title("Input waveform")
plt.tight_layout(); plt.show()

## 3. Duration check

The first rule. A recording outside the allowed length is rejected straight away —
there is no point cleaning audio that will be rejected anyway.

In [ ]:
duration_ok, duration_reason = check_duration(y_raw, sr, CONFIG["min_duration_s"], CONFIG["max_duration_s"])
print(f"Duration {len(y_raw) / sr:.2f} s  ->  {'OK' if duration_ok else 'FAIL'} ({duration_reason})")

## 4. Quality measurements before cleaning

`gate_inputs()` measures everything the gate models need:

| Measure | Meaning |
|---|---|
| `estimated_snr_db` | How loud the voice is compared with the noise. Uses silence-based SNR when the clip has pauses, otherwise HNR (normal for a sustained vowel). |
| `method` | Which of the two estimators was used. |
| `clipping_pct` | % of samples at full scale (distortion from recording too loud). |
| `voice_activity_pct` | % of the clip that contains sound. |
| `level_dbfs` | Overall loudness. Reported only, not used to pass/fail (differs between phones). |
| `mfcc_mean_*` | Spectral shape summary (used by the comp1-B models). |

In [ ]:
quality_before = gate_inputs(y_raw, sr, CONFIG)
main_keys = ["estimated_snr_db", "method", "clipping_pct", "voice_activity_pct", "level_dbfs", "duration_s"]
display(pd.Series({k: quality_before[k] for k in main_keys}, name="before cleaning").to_frame())

## 5. Enhancement (noise cleaning)

The audio is cleaned with the method set in the config (default: `noisereduce`
spectral gating). The waveform and spectrogram plots show raw (top) and cleaned
(bottom) on the same scale.

In [ ]:
y_enh = enhance(y_raw, sr, CONFIG["enhancement_method"], CONFIG["seed"])
quality_after = gate_inputs(y_enh, sr, CONFIG)

NB_DIR = PROJECT_ROOT / "outputs" / "component1_notebook"
NB_DIR.mkdir(parents=True, exist_ok=True)
before = (quality_before["estimated_snr_db"], quality_before["method"])
after = (quality_after["estimated_snr_db"], quality_after["method"])
display(Image(filename=str(plot_waveforms(y_raw, y_enh, sr, NB_DIR / "waveform.png", before=before, after=after))))
display(Image(filename=str(plot_spectrograms(y_raw, y_enh, sr, NB_DIR / "spectrogram.png", before=before, after=after))))

print("Raw:");      display(Audio(y_raw, rate=sr))
print("Cleaned:");  display(Audio(y_enh, rate=sr))

display(pd.DataFrame({"before": {k: quality_before[k] for k in main_keys},
                      "after":  {k: quality_after[k] for k in main_keys}}))

## 6. Biomarker preservation check

Parkinson's shows up in the voice as small irregularities (jitter, shimmer, lower HNR).
A noise filter can mistake those for noise and remove them — a PD voice could then
look healthy. So Component 1 measures the **receiving component's own features** before and after
cleaning — Component 2's for a vowel, Component 3's for DDK — and checks each stayed
within its allowed relative change.

- **All within tolerance** → the cleaned audio is used.
- **Any outside tolerance** → cleaning altered the biomarkers, so the **original** audio is used.

In [ ]:
extract_features, tolerance_key = TASK_FEATURES[TASK]   # same choice the pipeline makes
features_before = extract_features(y_raw, sr)
features_after = extract_features(y_enh, sr)
preservation = compare_features(features_before, features_after, CONFIG[tolerance_key])

rows = []
for name, tol in CONFIG[tolerance_key].items():
    change = preservation["per_feature"].get(name, float("nan"))
    rows.append({
        "feature": name,
        "before": None if features_before is None else features_before[name],
        "after": None if features_after is None else features_after[name],
        "relative change": change,
        "allowed": tol,
        "preserved": bool(change <= tol) if change == change else False,
    })
display(pd.DataFrame(rows).set_index("feature"))

use_enhanced = preservation["preserved"]
print("Biomarkers preserved:", use_enhanced)
print("Audio the gate will judge and forward:", "CLEANED" if use_enhanced else "ORIGINAL (raw)")

## 7. Gate decision

Every gate model first applies the same hard rules, in this order:
too short / too long → clipped → no voice. Only then does the selected model decide
(`comp1-A`: PASS if the SNR/HNR estimate ≥ the threshold).

In [ ]:
judged = quality_after if use_enhanced else quality_before
decision = GATE.predict(judged)

print(f"Gate model    : {GATE.name}")
print(f"Decision      : {decision['decision']}")
print(f"Reason        : {decision['reason']}")
print(f"Quality score : {decision['quality_score']:.1f} / 100")

## 8. Full pipeline in one call

`run_gate()` does sections 3–7 in one call — this is exactly what the CLI, the API and
the screening pipeline run. It also writes the output folder:

`outputs/component1/<clip_id>/` → `raw.wav`, `enhanced.wav`, `waveform.png`,
`spectrogram.png`, `report.json`.

In [ ]:
report = run_gate(INPUT_PATH, TASK, CONFIG, gate=GATE)

summary_keys = ["clip_id", "task", "gate_model", "decision", "reason", "quality_score",
                "audio_choice", "next_component", "user_message", "config_version"]
display(pd.Series({k: report[k] for k in summary_keys}, name="report.json").to_frame())

OUT_DIR = resolve_path(CONFIG, "outputs") / report["clip_id"]
print("Output folder:", OUT_DIR)
for f in sorted(OUT_DIR.iterdir()):
    print("  ", f.name)

## 9. Hand-off to Component 2

Component 1 must forward the audio **without changing what Component 2 sees**.
Component 2's zero-crossing and pitch features depend on sample rate, so the forwarded
file keeps the recording's own sample rate. This cell compares Component 2's prediction on
the original file and on the file Component 1 forwards — the two should match.

(Applies to `vowel` recordings that PASS.)

In [ ]:
from component2_phonation.predict import predict

if TASK == "vowel" and report["decision"] == "PASS":
    c2_model = resolve_path(CONFIG, "component2_model")
    forwarded = report["audio_for_next_stage"]
    original = predict(str(INPUT_PATH), model_path=str(c2_model))
    via_c1 = predict(forwarded, model_path=str(c2_model))
    display(pd.DataFrame({
        "sample rate (Hz)": [sr, sf.info(forwarded).samplerate],
        "Component 2 prediction": [original["prediction"], via_c1["prediction"]],
        "P(PD)": [round(original["probability_pd"], 3), round(via_c1["probability_pd"], 3)],
    }, index=["original recording", "forwarded by Component 1"]))
    print("Component 2 model:", c2_model.name)
else:
    print("Skipped: only for vowel recordings that PASS.")

## 10. Behaviour under different recording conditions

The same recording is damaged in controlled ways and passed through the full pipeline.
Expected: the clean recording passes; heavy noise, clipping, too short and too long are
rejected with a clear message for the user.

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])
COND_DIR = NB_DIR / "conditions"
COND_DIR.mkdir(parents=True, exist_ok=True)

short_len = int(2.0 * sr)
long_reps = int(np.ceil(CONFIG["max_duration_s"] * sr / len(y_raw))) + 1
conditions = {
    "clean (as recorded)": y_raw,
    "noise 20 dB": add_white_noise(y_raw, 20, rng),
    "noise 10 dB": add_white_noise(y_raw, 10, rng),
    "noise 5 dB": add_white_noise(y_raw, 5, rng),
    "noise 0 dB": add_white_noise(y_raw, 0, rng),
    "quiet (-12 dB gain)": apply_gain(y_raw, -12),
    "clipped (too loud)": np.clip(y_raw / (np.max(np.abs(y_raw)) + 1e-12) * 4.0, -1.0, 1.0),
    "too short (2 s)": y_raw[:short_len],
    "too long": np.tile(y_raw, long_reps),
}

rows = []
for name, y in conditions.items():
    path = COND_DIR / (name.split(" (")[0].replace(" ", "_") + ".wav")
    peak = np.max(np.abs(y))
    if name != "clipped (too loud)" and peak > 0.99:   # keep non-clipping cases below full scale
        y = 0.99 * y / peak
    sf.write(path, y, sr)
    r = run_gate(path, TASK, CONFIG, gate=GATE)
    q = r["quality_after"] if r["audio_choice"] == "enhanced" else r["quality_before"]
    rows.append({
        "condition": name,
        "duration (s)": round(r["duration_s"], 2),
        "SNR/HNR (dB)": None if q is None or q["estimated_snr_db"] is None else round(q["estimated_snr_db"], 1),
        "decision": r["decision"],
        "reason": r["reason"],
        "score": round(r["quality_score"], 1),
        "message to user": r["user_message"],
    })
display(pd.DataFrame(rows).set_index("condition"))

## 11. Evidence for the 10 dB threshold

From the robustness study (`experiment.py`, saved in
`results/component1_robustness/robustness_summary.csv`): clean recordings were degraded
with noise at known SNRs and passed to Component 2. Agreement with Component 2's
clean-audio prediction stays high at 10 dB and above, then drops sharply below it.
That is why the gate rejects recordings under 10 dB.

In [ ]:
summary_csv = resolve_path(CONFIG, "results") / "robustness_summary.csv"
robust = pd.read_csv(summary_csv)
display(robust)

noise = robust[robust["condition"].str.startswith("noise_snr_")].copy()
noise["snr_db"] = noise["condition"].str.extract(r"noise_snr_(-?\d+)db").astype(float)
clean_agreement = robust.loc[robust["condition"] == "clean", "prediction_agreement_with_clean"].iloc[0]
noise = noise.sort_values("snr_db", ascending=False)

plt.figure(figsize=(7, 4))
plt.plot(noise["snr_db"], noise["prediction_agreement_with_clean"], marker="o", label="noisy recordings")
plt.axhline(clean_agreement, linestyle=":", color="grey", label="clean recording")
plt.axvline(CONFIG["reliable_db"], linestyle="--", color="tab:red", label=f'gate threshold ({CONFIG["reliable_db"]:g} dB)')
plt.gca().invert_xaxis()
plt.xlabel("Injected noise level, SNR (dB) — lower is noisier")
plt.ylabel("Agreement with clean prediction")
plt.ylim(0, 1.05)
plt.title("Component 2 reliability vs noise level")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
enh_csv = resolve_path(CONFIG, "results") / "enhancement_summary.csv"
if enh_csv.is_file():
    enh = pd.read_csv(enh_csv)
    table = enh.pivot(index="condition", columns="variant", values="prediction_agreement_with_clean")
    display(table.round(3))
    table.plot.bar(figsize=(10, 4), rot=30)
    plt.ylabel("Agreement with clean prediction")
    plt.title("Does cleaning help Component 2? (degraded vs enhanced)")
    plt.ylim(0, 1.05); plt.tight_layout(); plt.show()
else:
    print("enhancement_summary.csv not found - run python -m component1_robustness.evaluate_enhancement")

## 12. Personal recordings (inference only)

Put your own recordings in `data/personal_recordings/` (`.wav`, `.m4a` or `.mp3`) and
set the task. Each one is run through the gate exactly as a real user upload would be.
These recordings are **never** used for training, thresholds or evaluation.

In [ ]:
PERSONAL_DIR = PROJECT_ROOT / "data" / "personal_recordings"
PERSONAL_TASK = "vowel"   # or "ddk"
PERSONAL_DIR.mkdir(parents=True, exist_ok=True)

personal = sorted(p for p in PERSONAL_DIR.iterdir() if p.suffix.lower() in (".wav", ".m4a", ".mp3"))
if not personal:
    print(f"No recordings in {PERSONAL_DIR} - add some and re-run this cell.")
rows = []
for path in personal:
    r = run_gate(path, PERSONAL_TASK, CONFIG, gate=GATE)
    rows.append({"file": path.name, "decision": r["decision"], "reason": r["reason"],
                 "score": round(r["quality_score"], 1), "forwarded": r["audio_choice"],
                 "message to user": r["user_message"]})
if rows:
    display(pd.DataFrame(rows).set_index("file"))

## Summary

- **Input:** one recording + task (`vowel` / `ddk`).
- **Process:** standardise (mono, original sample rate) → duration check → quality
  measurements → enhancement → biomarker preservation check → gate decision.
- **Output:** PASS/FAIL, quality score, user message, `report.json`, and the audio forwarded
  to Component 2/3 (cleaned only if the biomarkers survived cleaning).
- **Verification:** automated tests in `component1_robustness/tests`
  (`python -m pytest component1_robustness/tests -q`).

Known limitations are listed in `component1_robustness/README.md`.